In [6]:
from scipy.stats import norm
import numpy as np
import time
import import_ipynb
from Geometric_Brownian_Motion import simulate_GBM, simulate_GBM_antithetic
from Black_Scholes_Closed_Form import european_mc, european_mc_antithetic
#from Greeks import pathwise_delta, pathwise_vega, bs_delta, bs_vega
import matplotlib.pyplot as plt
from scipy.stats import qmc

In [9]:
def american_longstaff_schwartz(S0, K, r, sigma, T, n, n_paths, type="put"):
    t, S = simulate_GBM(S0, r, sigma, T, n, n_paths)
    dt = T / n
    
    if type == "put":
        payoff_fn = lambda s: np.maximum(K - s, 0)
    else:
        payoff_fn = lambda s: np.maximum(s - K, 0)
    
    # cash_flows[i] = the discounted-to-its-own-exercise-time cash flow for path i
    # exercise_time[i] = which timestep path i was exercised at (or n_steps if never)
    cash_flows = payoff_fn(S[:, -1])  # start by assuming exercise at expiry
    exercise_time = np.full(n_paths, n)
    
    # walk backward, skip the last column (already handled above) and t=0 (nothing to decide before start)
    for step in range(n - 1, 0, -1):
        S_t = S[:, step]
        exercise_value = payoff_fn(S_t)
        itm = exercise_value > 0
        
        if itm.sum() == 0:
            continue
        
        # discount the recorded cash flow back to THIS timestep for the regression target
        time_to_cashflow = (exercise_time[itm] - step) * dt
        Y = cash_flows[itm] * np.exp(-r * time_to_cashflow)
        X = S_t[itm]
        
        # regress Y on polynomial basis of X
        A = np.vstack([np.ones_like(X), X, X**2]).T
        coeffs, _, _, _ = np.linalg.lstsq(A, Y, rcond=None)
        continuation_value = A @ coeffs
        
        # decide: exercise now if it beats the estimated continuation value
        exercise_now = exercise_value[itm] > continuation_value
        
        itm_indices = np.where(itm)[0]
        exercise_indices = itm_indices[exercise_now]
        
        cash_flows[exercise_indices] = exercise_value[itm][exercise_now]
        exercise_time[exercise_indices] = step
    
    # discount every path's cash flow back to time 0 from its own exercise time
    discounted = cash_flows * np.exp(-r * exercise_time * dt)
    price = discounted.mean()
    se = discounted.std(ddof=1) / np.sqrt(n_paths)
    return price, se

In [10]:
S0, K, r, sigma, T = 100, 100, 0.05, 0.2, 1.0

am_put_price, am_se = american_longstaff_schwartz(S0, K, r, sigma, T, n=252, n_paths=100000, type="put")
eu_put_price, _ = european_mc(S0, K, r, sigma, T, n=252, n_paths=100000, type="put")

print(f"American put: {am_put_price:.4f} ± {1.96*am_se:.4f}")
print(f"European put: {eu_put_price:.4f}")

American put: 6.0637 ± 0.0438
European put: 5.5893


In [11]:
am_call_price, am_call_se = american_longstaff_schwartz(S0, K, r, sigma, T, n=252, n_paths=100000, type="call")
eu_call_price, _ = european_mc(S0, K, r, sigma, T, n=252, n_paths=100000, type="call")

print(f"American call: {am_call_price:.4f} ± {1.96*am_call_se:.4f}")
print(f"European call: {eu_call_price:.4f}")

American call: 10.4731 ± 0.0914
European call: 10.3876
